![Banner](../Image/04_DeepCuaslaML.png)

# 4.5 Graph Neural Network (GNN) Causal Models

This notebook introduces GNN-based causal modeling for multivariate time series and implements three practical model families:

1. **GVAR** (Graph Vector Autoregression)
2. **CausalGNN / CD-GNN** (joint graph learning + dynamics)
3. **CUTS+ inspired model** (causal discovery under missing data)

We use sector ETF returns as a realistic financial dataset and keep the implementation in pure PyTorch (no mandatory `torch-geometric`).

## Part 1: Theoretical Foundation

### Why GNNs for Causal Discovery?

A GNN treats variables as nodes and causal relationships as edges in a graph

$$\mathcal{G}=(\mathcal{V},\mathcal{E}).$$

Message passing makes the causal structure explicit and inspectable:

$$
\mathbf{h}_i^{(\ell+1)}=\mathrm{UPDATE}\!\left(\mathbf{h}_i^{(\ell)},\ \mathrm{AGGREGATE}\left(\{\mathbf{h}_j^{(\ell)}:j\in\mathcal{N}(i)\}\right)\right).
$$

Unlike RNNs/Transformers where causal effects are implicit in hidden states, in GNN causal models the graph itself is a first-class object that can be regularized, inspected, and interpreted.

## Part 2: The Three Models

### 1) GVAR — Graph Vector Autoregression

GVAR extends classical VAR with a learned graph and nonlinear feature transforms:

$$
X_i^{(t)}=\sum_{k=1}^{p}\sum_{j=1}^{d}A_{ij}^{(k)}\,\phi\!\left(X_j^{(t-k)}\right)+\epsilon_i^{(t)}.
$$

- $A^{(k)}\in\mathbb{R}^{d\times d}$: lag-specific adjacency (learned)
- $\phi(\cdot)$: nonlinear transform (GNN feature map)
- sparsity via L1/group-LASSO style penalties

### 2) CausalGNN / CD-GNN

Two-stage joint learning:

1. **Graph learner** outputs soft adjacency $\hat{G}$ with NOTEARS acyclicity

$$
h(\mathbf{A})=\mathrm{tr}(e^{\mathbf{A}\odot\mathbf{A}})-d=0
$$

2. **Dynamics learner** propagates messages along parents:

$$
\mathbf{m}_{j\rightarrow i}^{(t)}=\mathrm{MSG}(\mathbf{h}_j^{(t)},\mathbf{h}_i^{(t)},e_{ij}),\quad
\mathbf{h}_i^{(t+1)}=\mathrm{UPDATE}\!\left(\mathbf{h}_i^{(t)},\sum_{j\in\mathrm{Pa}(i)}\mathbf{m}_{j\rightarrow i}^{(t)}\right).
$$

### 3) CUTS / CUTS+

Designed for unevenly sampled or missing time series:

- causal-aware imputation network
- variational posterior over graphs: $q(G)\approx\prod_{ij}\mathrm{Bernoulli}(\pi_{ij})$
- joint optimization of reconstruction + imputation + graph regularization

A common ELBO form is:

$$
\mathcal{L}=\mathbb{E}[\log p(X_{\mathrm{obs}}\mid G,X_{\mathrm{miss}})]
+\mathbb{E}[\log p(X_{\mathrm{miss}}\mid G)]
-\mathrm{KL}[q(G)\|p(G)].
$$

### Setup and Imports

In [ ]:
import importlib
import subprocess
import sys

PACKAGES = [
    "numpy", "pandas", "scipy", "torch", "scikit-learn",
    "matplotlib", "seaborn", "networkx", "yfinance",
]

for pkg in PACKAGES:
    mod = "sklearn" if pkg == "scikit-learn" else pkg
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    import pydeepcausalml  # noqa: F401
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/zia207/PyDeepCausalML.git"]
    )

import pydeepcausalml
print("pydeepcausalml", pydeepcausalml.__version__, "ready.")


import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from pydeepcausalml import set_seed

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())

set_seed(42)
run_fast = True

import networkx as nx


**What this code does**

- Imports core packages for modeling (`torch`), data handling (`numpy`, `pandas`), plotting (`matplotlib`, `seaborn`), and data download (`yfinance`).
- Sets random seeds for reproducibility.
- Selects the compute device (`cuda` when available, otherwise `cpu`).

### Data Pipeline and Dataset Construction

In [ ]:
TICKERS = {
    "XLF": "Financials",
    "XLE": "Energy",
    "XLK": "Technology",
    "XLV": "Healthcare",
    "XLI": "Industrials",
    "XLY": "ConsumerDisc",
    "XLP": "ConsumerStap",
    "XLU": "Utilities",
}

import yfinance as yf

raw = yf.download(
    list(TICKERS.keys()),
    start="2018-01-01",
    end="2024-01-01",
    auto_adjust=True,
    progress=False,
)["Close"]

returns = np.log(raw / raw.shift(1)).dropna()
returns.columns = [TICKERS[t] for t in returns.columns]

if returns.shape[0] < 50:
    print("yfinance unavailable; using synthetic demo data.")
    rng = np.random.default_rng(42)
    T, d = 1500, len(TICKERS)
    VAR_NAMES = list(TICKERS.values())
    market = rng.normal(0.0, 0.8, size=(T, 1)).astype(np.float64)
    idio = rng.normal(0.0, 0.6, size=(T, d)).astype(np.float64)
    loading = np.linspace(0.5, 1.1, d, dtype=np.float64)[None, :]
    data_np = (market @ loading + idio).astype(np.float64)
else:
    data_np = returns.values.astype(np.float64)
    T, d = data_np.shape
    VAR_NAMES = returns.columns.tolist()

print(f"Shape: {data_np.shape}")
print(f"Variables ({d}): {VAR_NAMES}")
print(f"Time steps: {T}")

LAG = 5
EPOCHS = 50 if run_fast else 100
HIDDEN = 32
DEVICE = "cpu"


**What this code does**

- Downloads sector ETF prices and converts them to log returns.
- Standardizes variables so each series has comparable scale.
- Builds lagged supervised samples where inputs are the past `LAG` steps and targets are next-step values.
- Splits train/validation data and injects synthetic missingness for the CUTS+ demo.
- Creates `DataLoader` objects for both complete-data and missing-data training.

### Shared GNN Utilities

In [ ]:
from pydeepcausalml import GVAR, CausalGNN, CUTS, gnn_causal_model

def plot_graph(adj, names, title):
    G = nx.DiGraph()
    for i, tgt in enumerate(names):
        for j, src in enumerate(names):
            if adj[i, j] > 0.05:
                G.add_edge(src, tgt, weight=float(adj[i, j]))
    pos = nx.spring_layout(G, seed=42)
    plt.figure(figsize=(7, 6))
    nx.draw_networkx(G, pos, with_labels=True, node_size=800, font_size=8, arrows=True)
    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.show()


**What this code does**

- Defines `acyclicity_penalty(A)` (NOTEARS) to discourage cyclic graphs.
- Defines adjacency normalization to stabilize message passing.
- Adds a heatmap helper to visualize learned causal matrices.
- Implements a reusable training loop with early stopping, gradient clipping, and optional regularization losses.
- Implements validation MSE evaluation.

## Part 3: Model 1 — GVAR (Graph VAR)

This model learns lag-specific adjacency matrices jointly with nonlinear temporal dynamics.

### GVAR Model Definition

In [ ]:
# GVAR — Graph Vector Autoregression with lag-specific soft adjacency
print("GVAR: pydeepcausalml.timeseries.gnn_models.GVAR")


**What this code does**

- `GVARGraphLearner` learns lag-specific soft adjacency matrices with no self-loops.
- `GVARMessagePass` performs per-lag message passing and aggregates lag information.
- `GVAR` combines graph learning + temporal feature propagation to forecast next-step values.
- Returns prediction plus sparsity and DAG penalties to guide causal structure learning.

## Part 4: Model 2 — CausalGNN / CD-GNN

This model learns a soft DAG and uses edge-conditioned message passing for forecasting.

### CausalGNN / CD-GNN Definition

In [ ]:
# CausalGNN — bilinear graph learner with GRU encoder
print("CausalGNN: pydeepcausalml.timeseries.gnn_models.CausalGNN")


**What this code does**

- `CausalGraphLearner` infers a soft graph from temporal summaries.
- `EdgeConvLayer` computes edge-conditioned messages and updates node states via GRU-style updates.
- `CausalGNN` encodes temporal dynamics, propagates messages over the learned graph, and predicts next-step values.
- Adds sparsity and acyclicity penalties so structure learning stays interpretable.

## Part 5: Model 3 — CUTS+ Inspired (Missing Data)

A practical approximation of CUTS+ ideas: learn probabilistic graph edges and jointly impute missing values before forecasting.

### CUTS+ Inspired Model Definition

In [ ]:
# CUTS+ — variational Bernoulli graph posterior for irregular series
print("CUTS: pydeepcausalml.timeseries.gnn_models.CUTS")


**What this code does**

- Implements a lightweight CUTS+-style model for missing-data causal discovery.
- Learns probabilistic edges (`q(G)`) using Bernoulli parameters.
- Imputes missing values using an imputation network before temporal encoding.
- Performs graph message passing for prediction and regularizes with DAG + KL-to-sparse-prior terms.
- Includes a dedicated training loop for this composite objective.

## Part 6: Train All Models

The regularized objective is:

- **GVAR / CausalGNN:** `MSE + DAG penalty + sparsity`
- **CUTS+ inspired:** `MSE + DAG penalty + KL(graph posterior || prior)`

### Initialize and Train Models

In [ ]:
model_gvar = gnn_causal_model(
    "gvar", lag=LAG, hidden=HIDDEN, lambda_dag=0.1,
    epochs=EPOCHS, batch_size=64, lr=1e-3, device=DEVICE, random_state=42,
)
model_cgnn = gnn_causal_model(
    "causal_gnn", lag=LAG, hidden=HIDDEN,
    epochs=EPOCHS, batch_size=64, lr=1e-3, device=DEVICE, random_state=42,
)
model_cuts = gnn_causal_model(
    "cuts", lag=LAG, hidden=HIDDEN,
    epochs=EPOCHS, batch_size=64, lr=1e-3, device=DEVICE, random_state=42,
)

for label, est in [("GVAR", model_gvar), ("CausalGNN", model_cgnn), ("CUTS", model_cuts)]:
    print(f"Training {label} ...")
    est.fit(data_np)
    print(f"  done — loss={est.history_['loss'][-1]:.4f}")


**What this code does**

- Instantiates `GVAR`, `CausalGNN`, and `CUTSPlusLike` with chosen hyperparameters.
- Trains each model with its appropriate loss setup.
- Evaluates validation MSE so you can compare predictive performance across methods.

## Part 7: Visualize Learned Causal Graphs

### Plot Learned Causal Graphs

In [ ]:
C_gvar = model_gvar.causal_matrix()
C_cgnn = model_cgnn.causal_matrix()
C_cuts = model_cuts.causal_matrix()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (name, mat) in zip(axes, [("GVAR", C_gvar), ("CausalGNN", C_cgnn), ("CUTS", C_cuts)]):
    sns.heatmap(mat, ax=ax, cmap="magma", xticklabels=VAR_NAMES, yticklabels=VAR_NAMES)
    ax.set_title(f"{name} causal matrix")
plt.tight_layout()
plt.show()

plot_graph(C_gvar, VAR_NAMES, "GVAR learned graph")
plot_graph(C_cgnn, VAR_NAMES, "CausalGNN learned graph")
plot_graph(C_cuts, VAR_NAMES, "CUTS learned graph")


**What this code does**

- Extracts learned causal matrices from all three models.
- Plots heatmaps to compare learned directed dependency strengths.
- Prints top weighted edges (`source -> target`) to support interpretation.

## Notes and Extensions

- The CUTS+ implementation here is intentionally lightweight for teaching; full CUTS+ includes richer uneven-sampling handling and a tighter variational treatment.
- You can improve robustness by using rolling-window validation and bootstrapped edge confidence.
- For larger systems, add thresholding and stability selection before interpreting edges as causal hypotheses.
- Causal discovery from observational finance data is sensitive to confounders; use domain checks and intervention-style validation where possible.

## Summary and Conclusions

In this notebook, we explored neural network-based approaches for causal discovery in financial time series.

**Key steps and findings:**
- We simulated or loaded multivariate time series and constructed three causal discovery models:
    1. **GVAR (Graphical VAR):** A sparse linear baseline estimating Granger-causal dependencies via a regularized VAR.
    2. **CausalGNN:** A neural architecture learning directed dependencies in a data-driven manner, potentially modeling nonlinearity.
    3. **CUTS+-like:** A lightweight, attention-based model inspired by the CUTS+ method (here for teaching), which leverages neural attention to infer causal structure.
- For each model, we visualized the learned adjacency (causal) matrices and compared their dominant dependencies via heatmaps.
- We further extracted and printed the top-weighted causal edges, making the results interpretable and highlighting important directed relationships.

**Conclusions:**
- Neural causal discovery methods like CausalGNNs can flexibly recover both linear and nonlinear dependencies, offering richer modeling power than purely linear baselines.
- Visualization and ranking of discovered edges can help practitioners interpret learned causal graphs and guide further hypothesis testing or domain validation.
- For robust application in real finance data, additional techniques (regularization, thresholding, validation, bootstrapping) are essential to avoid spurious findings.

**Next steps** could include extending these models to larger systems, integrating domain priors, applying them to empirical financial datasets, and evaluating discovery stability under perturbation or intervention.

## Resources and Further Reading

- **Graph Neural Networks and Causal Discovery**
    - [Exploring Causal Learning through Graph Neural Networks: An In-depth Review](https://arxiv.org/html/2311.14994)
    - [Causal Discovery Toolbox (CDT) library](https://fentechsolutions.github.io/CausalDiscoveryToolbox/html/index.html)
    - [DoWhy: An End-to-End Library for Causal Inference](https://github.com/py-why/dowhy)
- **Financial Time Series Causality**
    - [Granger Causality Explained](https://en.wikipedia.org/wiki/Granger_causality)
    - [VAR Models for Financial Time Series](https://www.stat.pitt.edu/stoffer/tsa4/)
- **Machine Learning for Causal Inference**
    - [Elements of Causal Inference (Book)](https://mitpress.mit.edu/9780262037310/elements-of-causal-inference/)
- **Recent Advances and Tutorials**
    - [CUTS+: High-dimensional Causal Discovery from Irregular Time-series](https://arxiv.org/abs/2305.05890)
 
 These resources provide a deeper dive into both foundational theory and modern methods for neural and statistical causal discovery, especially in time series and financial domains.